# NeuroBridge Quickstart (Colab)

Use Runtime > Run all to execute this notebook top-to-bottom.

Links: [GitHub](https://github.com/yourusername/neurobridge) | [HuggingFace Space](https://huggingface.co/spaces/yourusername/neurobridge) | [PyPI](https://pypi.org/project/neurobridge/) | [Discord](https://discord.gg/neurobridge)

## 1) Installation
Install NeuroBridge in this Colab runtime.

In [ ]:
!pip install neurobridge

## 2) Basic Usage Across 5 Profiles
Adapt the same source text with all built-in profiles.

In [ ]:
from neurobridge import NeuroBridge, Profile

source_text = "This is urgent. You must finish the report immediately and provide a detailed breakdown of all budget percentages and assumptions."
bridge = NeuroBridge()

for profile in [Profile.ADHD, Profile.AUTISM, Profile.DYSLEXIA, Profile.ANXIETY, Profile.DYSCALCULIA]:
    bridge.set_profile(profile)
    result = bridge.chat(source_text)
    print(f"\n=== {profile.value} ===")
    print(result.adapted_text)

## 3) OpenAI Integration
Use a Colab secret named OPENAI_API_KEY if available.

In [ ]:
from neurobridge import Profile
from neurobridge.integrations.openai import wrap

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

if api_key:
    from openai import OpenAI
    client = wrap(OpenAI(api_key=api_key), profile=Profile.DYSLEXIA)
    print('OpenAI client wrapped. Call client.chat.completions.create(...) as usual.')
else:
    print('OPENAI_API_KEY not found in Colab secrets. Skipping live call.')

## 4) LangChain Integration
Wrap an LLM output with NeuroBridgeOutputParser.

In [ ]:
from neurobridge import Profile
from neurobridge.integrations.langchain import NeuroBridgeOutputParser

parser = NeuroBridgeOutputParser(profile=Profile.ADHD)
raw = "This explanation includes many details, dependencies, caveats, and urgency markers that can overload focus quickly."
adapted = parser.parse(raw)
print(adapted)

## 5) ProfileQuiz API
Score quiz answers and inspect the recommended profile.

In [ ]:
from neurobridge.core.quiz import QuizEngine

engine = QuizEngine()
answers = {q.id: 0 for q in engine.QUESTIONS}
result = engine.score_answers(answers)
print('Primary profile:', result.primary_profile.value)
print('Secondary profile:', result.secondary_profile.value if result.secondary_profile else None)
print('Confidence:', round(result.confidence * 100, 2), '%')

## 6) REST API Server
Start the NeuroBridge FastAPI server in the background and send an adapt request.

In [ ]:
import subprocess
import time
import requests

server = subprocess.Popen(['python', '-m', 'uvicorn', 'neurobridge.server.app:app', '--host', '127.0.0.1', '--port', '8000'])
time.sleep(2)

payload = {
    'text': 'This is critical and must be completed ASAP. Revenue was 22% lower than target.',
    'profile': 'anxiety'
}
resp = requests.post('http://127.0.0.1:8000/api/v1/adapt', json=payload, timeout=10)
print(resp.status_code)
print(resp.json())

server.terminate()
server.wait(timeout=5)

## 7) Custom Profile Builder
Define custom adaptation preferences and run a transformation.

In [ ]:
from neurobridge import CustomProfile, NeuroBridge

custom = CustomProfile(
    chunk_size=2,
    tone='calm',
    ambiguity_resolution='explicit',
    number_format='contextual',
    leading_style='summary_first',
    reading_level=6,
    max_sentence_words=12,
)

bridge = NeuroBridge()
bridge.set_profile(custom)
result = bridge.chat('Please provide a detailed update on architecture risks and deployment timing assumptions.')
print(result.adapted_text)

## 8) Performance Benchmark
Measure adaptation time for short, medium, and long input lengths.

In [ ]:
from time import perf_counter
from neurobridge import NeuroBridge, Profile

bridge = NeuroBridge()
bridge.set_profile(Profile.ADHD)

samples = {
    'short': ' '.join(['token'] * 80) + '.',
    'medium': ' '.join(['token'] * 500) + '.',
    'long': ' '.join(['token'] * 2000) + '.',
}

for label, text in samples.items():
    start = perf_counter()
    _ = bridge.adapt(text)
    elapsed_ms = (perf_counter() - start) * 1000
    print(f"{label}: {elapsed_ms:.2f} ms")